# Section 5.2 — Deterministic versus Stochastic Planning

Reproduces **Table 3** from the thesis.  
Runs DE (Deterministic Equivalent), WS (Wait-and-See), and EEV (Expected Value of the EV solution) on 4-cage, 8-cage, and 12-cage instances — 9 runs in total.


In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'models'))

import time
import numpy as np
import pandas as pd

from instance import (
    T, loc_mab, regional_mab,
    units_df as full_units_df,
    build_scenarios,
)
from IP import SalmonFarmingMILP
from DE import DeterministicEquivalent


In [2]:
# WS helper

def run_ws_for(units_df_sub, mip_gap=0.02):
    """Wait-and-See: solve each of 81 scenarios independently."""
    scenarios = build_scenarios()
    n_feas = 0
    ws_obj = 0.0
    t0 = time.time()

    for sc_idx, (sc_name, temps_sc, S_sc, prob) in enumerate(scenarios):
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
            scenario_name=sc_name,
        )
        milp.model.Params.OutputFlag = 0
        milp.model.Params.MIPGap = mip_gap
        milp.model.optimize()
        if milp.model.SolCount > 0:
            n_feas += 1
            ws_obj += prob * milp.model.ObjVal

    return ws_obj, time.time() - t0, n_feas


In [3]:
# EEV helper

NA_MONTHS = set(range(0, 45))  # stages 0-2 are fixed


def run_eev_for(units_df_sub, mip_gap=0.02):
    """
    EEV: solve EV (deterministic, expected-parameter) model, fix its stage 0-2
    decisions (stk, harv, h_exist, q), then evaluate across all 81 scenarios.
    """
    # Step 1: EV model (expected temperatures, blended survival)
    temps_exp = np.tile(
        np.array([5, 5, 5, 6, 9, 12, 14, 16.5, 15.5, 13, 10, 7.5]),
        (T // 12) + 1
    )[:T]
    S_exp = np.full(T, (2.0 * (1.0 - 0.0002) ** 30 + (1.0 - 0.001) ** 30) / 3.0)

    t0 = time.time()
    ev_m = SalmonFarmingMILP(
        units_df=units_df_sub, temps_t=temps_exp, survival_rates=S_exp,
        horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
        density_limit=25.0, verbose=False,
    )
    ev_m.model.Params.OutputFlag = 0
    ev_m.model.Params.MIPGap = mip_gap
    ev_m.model.optimize()
    if ev_m.model.SolCount == 0:
        print('  EV model infeasible')
        return None, time.time() - t0, 0

    # Step 2: Extract stage 0-2 decisions
    ev_stk = {
        (u, t): round(ev_m.variables['z'][u, t].X)
        for u in ev_m.U for t in ev_m.Tset if t in NA_MONTHS
    }
    ev_harv = {
        (u, s, t): round(ev_m.variables['h'][u, s, t].X)
        for u in ev_m.U for s in ev_m.Tset
        for t in ev_m.H_by_us.get((u, s), []) if t in NA_MONTHS
    }
    ev_hexist = {
        (u, t): round(ev_m.variables['h_exist'][u, t].X)
        for u in ev_m.U_exist for t in ev_m.Tset if t in NA_MONTHS
    }
    ev_q = {
        (u, s): ev_m.variables['q'][u, s].X
        for u in ev_m.U for s in ev_m.Tset
        if s in NA_MONTHS and round(ev_m.variables['z'][u, s].X) == 1
    }

    # Step 3: Fix EV decisions and evaluate on all 81 scenarios
    scenarios = build_scenarios()
    n_feas = 0
    eev_obj = 0.0
    feas_prob = 0.0

    for sc_idx, (sc_name, temps_sc, S_sc, prob) in enumerate(scenarios):
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab, regional_mab=regional_mab,
            scenario_name=sc_name, disable_economic_presolve=True,
        )
        model = milp.model
        model.update()

        for (u, t), val in ev_stk.items():
            v = model.getVarByName(f'z[{u},{t}]')
            if v is not None:
                v.LB = val
                v.UB = val
        for (u, s, t), val in ev_harv.items():
            v = model.getVarByName(f'h[{u},{s},{t}]')
            if v is not None:
                v.LB = val
                v.UB = val
        for (u, t), val in ev_hexist.items():
            v = model.getVarByName(f'h_exist[{u},{t}]')
            if v is not None:
                v.LB = val
                v.UB = val
        for (u, s), val in ev_q.items():
            v = model.getVarByName(f'q[{u},{s}]')
            if v is not None:
                v.LB = val
                v.UB = val

        model.Params.OutputFlag = 0
        model.Params.MIPGap = mip_gap
        model.update()
        model.optimize()

        if model.SolCount > 0:
            n_feas += 1
            feas_prob += prob
            eev_obj += prob * model.ObjVal

    eev_normalized = eev_obj / feas_prob if feas_prob > 0 else 0.0
    return eev_normalized, time.time() - t0, n_feas


In [ ]:
# Main experiment: {DE, WS, EEV} x {4, 8, 12 cages}

results = []

for n_cages in [4, 8, 12]:
    mip_gap = 0.035 if n_cages == 12 else 0.02
    sub_df  = full_units_df.head(n_cages).reset_index(drop=True)

    print(f'\n{"="*60}')
    print(f'  n_cages={n_cages}   mip_gap={mip_gap:.1%}')
    print(f'{"="*60}')

    # DE
    print(f'\n--- DE ({n_cages} cages) ---')
    de = DeterministicEquivalent(
        units_df=sub_df, loc_mab=loc_mab, regional_mab=regional_mab,
        T=T, mip_gap=mip_gap,
    )
    de.build()
    de.solve()
    de_obj  = de.obj_val
    de_time = de.solve_time
    de_gap  = de.model.MIPGap if de.model.SolCount > 0 else float('nan')
    print(f'  DE  = {de_obj/1e6:.3f} MNOK  |  {de_time:.1f}s  |  gap {de_gap:.2%}')

    # WS
    print(f'\n--- WS ({n_cages} cages) ---')
    ws_obj, ws_time, ws_feas = run_ws_for(sub_df, mip_gap=mip_gap)
    print(f'  WS  = {ws_obj/1e6:.3f} MNOK  |  {ws_time:.1f}s  |  {ws_feas}/81 feasible')

    # EEV
    print(f'\n--- EEV ({n_cages} cages) ---')
    eev_obj, eev_time, eev_feas = run_eev_for(sub_df, mip_gap=mip_gap)
    print(f'  EEV = {eev_obj/1e6:.3f} MNOK  |  {eev_time:.1f}s  |  {eev_feas}/81 feasible')

    vss  = (de_obj - eev_obj) / 1e6
    evpi = (ws_obj - de_obj)  / 1e6
    print(f'  VSS = {vss:.3f} MNOK   EVPI = {evpi:.3f} MNOK')

    results.append({
        'n_cages':      n_cages,
        'DE [MNOK]':    round(de_obj  / 1e6, 3),
        'WS [MNOK]':    round(ws_obj  / 1e6, 3),
        'EEV [MNOK]':   round(eev_obj / 1e6, 3),
        'VSS [MNOK]':   round(vss,  3),
        'EVPI [MNOK]':  round(evpi, 3),
        'DE time [s]':  round(de_time,  1),
        'WS time [s]':  round(ws_time,  1),
        'EEV time [s]': round(eev_time, 1),
        'MIP gap':      mip_gap,
    })



  n_cages=4   mip_gap=2.0%

--- DE (4 cages) ---
Set parameter Username
Set parameter LicenseID to value 2786519
Academic license - for non-commercial use only - expires 2027-03-03
  Prepared 81 scenarios  (3×3×3×3, 4 stages of uncertainty)
  Variables: 0
  Per-scenario constraints: 0
  Total constraints with NAC: 0
Set parameter MIPGap to value 0.02

DE model built in 44.1s  | 0 vars  | 0 constraints

--- Solving DE ---
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 25.3.0 25D2128)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1387381 rows, 894807 columns and 9118304 nonzeros
Model fingerprint: 0x2f617856
Variable types: 638523 continuous, 236844 integer (236844 binary)
Semi-Variable types: 19440 continuous, 0 integer
Coefficient statistics:
  Matrix range     [1e-01, 2e+07]
  Objective range  [2e-03, 2e+05]
  Bounds range     [1e+00, 3e+06]
  RHS range        [1e+00, 6e+06]
Presolve remo

In [5]:
# Results table (Table 3)

df_results = pd.DataFrame(results).set_index('n_cages')
display(df_results)


,DE [MNOK],WS [MNOK],EEV [MNOK],VSS [MNOK],EVPI [MNOK],DE time [s],WS time [s],EEV time [s],MIP gap
n_cages,,,,,,,,,
4,321.145,370.176,278.859,42.287,49.030,13.2,34.7,23.3,0.02
8,551.984,643.691,475.996,75.988,91.707,97.4,109.1,46.8,0.02
12,720.038,813.519,650.816,69.222,93.481,177.1,133.9,72.1,0.10
